# Transformerによる計算機作成

---
## 目的

Transformerの構造について理解する．

## モジュールのインポートとGPUの確認
はじめに必要なモジュールをインポートする．
そして，GPUが使用可能かどうかを確認する．

In [ ]:
import os
import numpy as np
import math
from time import time
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import gdown

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Use device:', device)

## データローダの作成
まず，データローダを用意します．データは0から9までの数字と加算記号，開始，終了のフラグです．また，３桁の数字の足し算を行うため，各桁の値を１つずつランダムに生成して連結しています．


In [ ]:
word2id = {str(i): i for i in range(10)}
word2id.update({"<pad>": 10, "+": 11, "<eos>": 12})
id2word = {v: k for k, v in word2id.items()}


class CalcDataset(torch.utils.data.Dataset):

    def transform(self, string, seq_len=7):
        tmp = []
        for i, c in enumerate(string):
            try:
                tmp.append(word2id[c])
            except:
                tmp += [word2id["<pad>"]] * (seq_len - i)
                break
        return tmp

    def __init__(self, data_num, train=True):
        super().__init__()
        self.data_num = data_num
        self.train = train
        self.data = []
        self.label = []

        for _ in range(data_num):
            x = int("".join([random.choice(list("0123456789")) for _ in range(random.randint(1, 3))] ))
            y = int("".join([random.choice(list("0123456789")) for _ in range(random.randint(1, 3))] ))
            left = ("{:*<7s}".format(str(x) + "+" + str(y))).replace("*", "<pad>")
            self.data.append(self.transform(left))

            z = x + y
            right = ("{:*<6s}".format(str(z))).replace("*", "<pad>")
            right = self.transform(right, seq_len=5)
            right = [12] + right
            right[right.index(10)] = 12
            self.label.append(right)

        self.data = np.asarray(self.data)
        self.label = np.asarray(self.label)

    def __getitem__(self, item):
        d = self.data[item]
        l = self.label[item]
        return d, l

    def __len__(self):
        return self.data.shape[0]

## Transformerの実装

Transformer（[Vaswani et al., 2017](https://arxiv.org/abs/1706.03762)）は，CNNやRNNを使わずAttention機構のみで構成されたEncoder-Decoderモデルです．RNNのように逐次処理する必要がないため並列計算がしやすく，系列内の離れた要素間の依存関係も捉えやすいという特徴があります．

Encoderは，Multi-Head AttentionとFeed Forwardのブロックを$N$回スタックした構造です．Decoderはそれに加えてMasked Multi-Head Attentionを持ちます．学習時は全ターゲットを一度に入力しますが，未来の情報を参照できないようMaskをかけて予測します．評価時は，予測した単語を1つずつ次の入力に使う自己回帰的な生成を行います．

![transformer](images/transformer.png)

### Attentionの種類

- **Self-Attention**：Query，Key，Valueすべてに同じ系列を使うAttentionです．文中の単語同士の関係を捉えるために，EncoderとDecoderの両方で使われます．
- **Source-Target-Attention**：QueryにDecoder側の系列，Key・ValueにEncoderの出力を使うAttentionです．Decoderが，出力すべき単語を予測する際にEncoderの入力のどこに着目すべきかを学習します．

![self-st-att](images/self-and-st-attentions.png)

#### Scaled Dot-Product Attention

Self-Attentionの中身であり，QueryとKeyの内積から類似度を計算し，softmaxで正規化した重みをValueに掛けて出力します．

\begin{equation}
{\rm Attention}(Q, K, V)={\rm softmax} \left( \frac{QK^{T}}{\sqrt{d_k}} \right) V
\end{equation}

$d_k$はQueryの次元数です．内積の値が大きくなるとsoftmaxの勾配が消失しやすくなるため，$\sqrt{d_k}$で割ってスケーリングします．また，softmaxをかける前にMask（後述）を適用し，paddingや未来の情報に対するAttention weightがほぼ0になるようにします．

#### Multi-Head Attention

Query，Key，Valueを複数（$h$個）の小さな次元に分割し，それぞれでScaled Dot-Product Attentionを計算してから結合し，Linear層に通します．分割することで，異なる部分空間の情報に着目したAttentionを同時に学習できます．

![mh-att](images/scaled-mh-attentions.png)

### Mask

学習はミニバッチ単位で行うため，バッチ内で系列長を揃えるためにpaddingを行います．そのままAttentionを計算するとpadding部分にもAttention weightが計算されてしまうため，Maskを使って防ぎます．

#### Encoderに対するMask

Multi-Head Attention内で，padding位置のAttention weightがほぼ0になるようMaskをかけます．

#### Decoderに対するMask

Decoderは学習時に全ターゲットを一度に入力しますが，ある時刻の予測で未来の単語を参照できてしまうと，自己回帰的な生成（推論時の挙動）と矛盾します．そのため，各時刻より先の情報を参照できないようMaskをかけます．このMaskはDecoderのMasked Multi-Head Attentionで使われます．

![enc_dec_mask](images/enc_dec_mask.png)

In [ ]:
def enc_mask(batch_size, src, size):
    mask = src == word2id["<pad>"]
    mask = mask.float().masked_fill(mask == 1, float(0.0)).masked_fill(mask == 0, float(1.0))
    return mask.view(mask.size(0), 1, mask.size(1))

def dec_mask(batch_size, size):
    mask = torch.triu(torch.ones(size, size), 1)
    mask = mask.float().masked_fill(mask == 0, float(1.0)).masked_fill(mask == 1, float(0.0))
    mask = mask.view(1, *mask.shape)
    mask = mask.expand(batch_size, *mask.shape[1:])
    return mask

def create_masks(batch_size, src, trg):
    src_mask = enc_mask(batch_size, src, src.size(1))

    if trg is not None:
        size = trg.size(1)
        np_mask = dec_mask(batch_size, size)
        trg_mask = np_mask

    else:
        trg_mask = None
    return src_mask, trg_mask

### Multi-Head AttentionとSelf-Attention

In [ ]:
def attention(q, k, v, d_k, mask=None, dec_mask=False):
    scores = torch.matmul(q, k.transpose(-2, -1)) /  math.sqrt(d_k)
    if mask is not None:
        if dec_mask:
            mask = mask.view(mask.size(0), 1, mask.size(1), mask.size(2))
        else:
            mask = mask.unsqueeze(1)
        scores = scores.masked_fill(mask == 0, -1e9)

    scores = F.softmax(scores, dim=-1)

    output = torch.matmul(scores, v)
    return output


class MultiHeadAttention(nn.Module):
    def __init__(self, heads, embedding_dim):
        super().__init__()

        self.embedding_dim = embedding_dim
        self.d_k = embedding_dim // heads
        self.h = heads

        self.q_linear = nn.Linear(embedding_dim, embedding_dim)
        self.v_linear = nn.Linear(embedding_dim, embedding_dim)
        self.k_linear = nn.Linear(embedding_dim, embedding_dim)

        self.out = nn.Linear(embedding_dim, embedding_dim)

    def forward(self, q, k, v, mask=None, dec_mask=False):

        bs = q.size(0)
        k = self.k_linear(k).view(bs, -1, self.h, self.d_k)
        q = self.q_linear(q).view(bs, -1, self.h, self.d_k)
        v = self.v_linear(v).view(bs, -1, self.h, self.d_k)

        k = k.transpose(1, 2)
        q = q.transpose(1, 2)
        v = v.transpose(1, 2)

        scores = attention(q, k, v, self.d_k, mask, dec_mask)

        concat = scores.transpose(1,2).contiguous().view(bs, -1, self.embedding_dim)
        output = self.out(concat)

        return output

### FeedForwardNetwork

In [ ]:
class FeedForward(nn.Module):

    def __init__(self, embedding_dim, d_ff=2048):
        super().__init__()
        self.linear_1 = nn.Linear(embedding_dim, d_ff)
        self.linear_2 = nn.Linear(d_ff, embedding_dim)

    def forward(self, x):
        x = F.relu(self.linear_1(x))
        x = self.linear_2(x)
        return x

### Encoder-DecoderのLinear処理

In [ ]:
class EncoderLayer(nn.Module):

    def __init__(self, embedding_dim, heads):
        super().__init__()
        self.norm_1 = nn.LayerNorm(embedding_dim)
        self.norm_2 = nn.LayerNorm(embedding_dim)
        self.attn = MultiHeadAttention(heads, embedding_dim)
        self.ff = FeedForward(embedding_dim)

    def forward(self, x, mask):
        x2 = self.norm_1(x)
        x = x + self.attn(x2,x2,x2,mask, dec_mask=False)
        x2 = self.norm_2(x)
        x = x + self.ff(x2)
        return x

class DecoderLayer(nn.Module):

    def __init__(self, embedding_dim, heads):
        super().__init__()
        self.norm_1 = nn.LayerNorm(embedding_dim)
        self.norm_2 = nn.LayerNorm(embedding_dim)
        self.norm_3 = nn.LayerNorm(embedding_dim)

        self.attn_1 = MultiHeadAttention(heads, embedding_dim)
        self.attn_2 = MultiHeadAttention(heads, embedding_dim)
        self.ff = FeedForward(embedding_dim)

    def forward(self, x, e_outputs, src_mask, trg_mask):
        x2 = self.norm_1(x)
        x = x + self.attn_1(x2, x2, x2, trg_mask, dec_mask=True)
        x2 = self.norm_2(x)
        x = x + self.attn_2(x2, e_outputs, e_outputs, src_mask, dec_mask=False)
        x2 = self.norm_3(x)
        x = x + self.ff(x2)
        return x

### Encoder-DecoderのPositional Embedding (Positional Encoding) 処理

In [ ]:
class PositionalEncoder(nn.Module):

    def __init__(self, embedding_dim, max_seq_len = 200):
        super().__init__()
        self.embedding_dim = embedding_dim
        pe = torch.zeros(max_seq_len, embedding_dim)
        for pos in range(max_seq_len):
            for i in range(0, embedding_dim, 2):
                pe[pos, i] = \
                math.sin(pos / (10000 ** ((2 * i)/embedding_dim)))
                pe[pos, i + 1] = \
                math.cos(pos / (10000 ** ((2 * (i + 1))/embedding_dim)))
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x * math.sqrt(self.embedding_dim)
        seq_len = x.size(1)
        pe = self.pe[:, :seq_len]
        x = x + pe
        return x

### Transformerモデル

In [ ]:
class Encoder(nn.Module):

    def __init__(self, vocab_size, embedding_dim, N, heads):
        super().__init__()
        self.N = N
        self.embed = nn.Embedding(vocab_size, embedding_dim, padding_idx=word2id["<pad>"])
        self.pe = PositionalEncoder(embedding_dim)
        self.layers = nn.ModuleList([EncoderLayer(embedding_dim, heads) for i in range(N)]) # N個のEncoder Layerをリスト形式で追加
        self.norm = nn.LayerNorm(embedding_dim)

    def forward(self, src, mask):
        x = self.embed(src)
        x = self.pe(x)
        for i in range(self.N):
            x = self.layers[i](x, mask)
        return self.norm(x)


class Decoder(nn.Module):

    def __init__(self, vocab_size, embedding_dim, N, heads):
        super().__init__()
        self.N = N
        self.embed =  nn.Embedding(vocab_size, embedding_dim, padding_idx=word2id["<pad>"])
        self.pe = PositionalEncoder(embedding_dim)
        self.layers = nn.ModuleList([DecoderLayer(embedding_dim, heads) for i in range(N)])
        self.norm = nn.LayerNorm(embedding_dim)

    def forward(self, trg, e_outputs, src_mask, trg_mask):
        x = self.embed(trg)
        x = self.pe(x)
        for i in range(self.N):
            x = self.layers[i](x, e_outputs, src_mask, trg_mask)
        return self.norm(x)


class Transformer(nn.Module):

    def __init__(self, vocab_size, embedding_dim, N, heads):
        super().__init__()
        self.encoder = Encoder(vocab_size, embedding_dim, N, heads)
        self.decoder = Decoder(vocab_size, embedding_dim, N, heads)
        self.out = nn.Linear(embedding_dim, vocab_size)

    def forward(self, src, trg, src_mask, trg_mask):
        e_outputs = self.encoder(src, src_mask)
        d_output = self.decoder(trg, e_outputs, src_mask, trg_mask)
        output = self.out(d_output)
        return output

## 学習

デフォルトの設定で約1時間ほどかかります．特徴ベクトルの次元数が$embedding\_dim$，Multi-head Attentionのhead数が$heads$，層数が$n\_layers$です．

最適化手法は，Adam（$\beta_1=0.9, \beta_2=0.98$, $\varepsilon=10^{-9}$）に，学習初期に学習率を線形に増加させ，その後緩やかに減衰させるウォームアップ付きスケジューリングを組み合わせています．固定の学習率でAdamのみを使うと学習が不安定になり，精度が上がりにくいためです．

In [ ]:
# データセットの準備
batch_size = 100
epoch_num = 1000
train_data = CalcDataset(data_num=50000)
train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=16, pin_memory=True)

# Transformerの準備
embedding_dim = 512
n_layers = 6
heads = 8
vocab_size = len(word2id)
model = Transformer(vocab_size, embedding_dim, n_layers, heads).to(device)

# Optimizer: Adam + warmupありの学習率スケジューリング
warmup_steps = 4000
optimizer = optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)

def lr_lambda(step):
    step = max(step, 1)
    return (embedding_dim ** -0.5) * min(step ** -0.5, step * warmup_steps ** -1.5)

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

# 誤差関数
criterion = nn.CrossEntropyLoss(ignore_index=word2id["<pad>"])

In [ ]:
all_losses = []
start = time()
for epoch in range(1, epoch_num+1):
    epoch_loss = 0
    for src, trg in train_loader:
        model.zero_grad()

        src = src.to(device)
        trg = trg.to(device)

        trg_input = trg[:, :-1]
        src_mask, trg_mask = create_masks(batch_size, src, trg_input)
        src_mask = src_mask.to(device)
        trg_mask = trg_mask.to(device)

        preds = model(src, trg_input, src_mask, trg_mask)
        loss = criterion(preds.view(-1, preds.size(-1)), trg[:, 1:].contiguous().view(-1))

        loss.backward()
        epoch_loss += loss.item()

        optimizer.step()
        scheduler.step()

    elapsed_time = time() - start
    all_losses.append(epoch_loss)
    if epoch % 10 == 0:
        print("epoch: {}, mean loss: {:.4f}, lr: {:.6f}, elapsed_time: {:.4f}".format(epoch, epoch_loss / len(train_loader), scheduler.get_last_lr()[0], elapsed_time))

    if epoch % 100 == 0:
        model_name = "transformer_calculator_v{}.pt".format(epoch)
        torch.save({'model': model.state_dict()}, model_name)

## 評価

本来のTransformerはビームサーチで行っているが，今回は貪欲法で実装します．

In [ ]:
if not os.path.exists('./transformer_calculator_v1000.pt'):
    gdown.download(id='1n3zEPW1Et8HexBcpKIt0vppw5gNxoRDn', output='transformer_calculator_v1000.pt', quiet=False)

In [ ]:
batch_size = 1
test_data = CalcDataset(data_num = 50)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=True)
model = Transformer(vocab_size, embedding_dim, n_layers, heads).to(device)
model_name = "transformer_calculator_v{}.pt".format(1000)
checkpoint = torch.load(model_name, map_location=device)
model.load_state_dict(checkpoint["model"])

accuracy = 0

# 評価の実行
with torch.no_grad():
    for src, trg in test_loader:
        src = src.to(device)
        trg = trg.to(device)

        trg_input = trg[:, :].clone()
        src_mask, trg_mask = create_masks(batch_size, src, trg_input)
        src_mask = src_mask.to(device)
        trg_mask = trg_mask.to(device)

        # encoder
        e_output = model.encoder(src, src_mask)

        # decoder
        right = []
        for s in range(7):
            outputs = trg_input[:, :s+1]
            trg_mask_ = trg_mask[:, :s+1, :s+1]
            out = model.out(model.decoder(outputs, e_output, src_mask, trg_mask_))
            out = F.softmax(out, dim=2)

            if s == 0:
              index = torch.argmax(out.cpu().detach()).item()
            else:
              index = torch.argmax(out, dim=2)[0, -1].cpu().detach().item()
            token = id2word[index]

            if token == "<eos>":
                break
            right.append(token)

            trg_input[:, s+1] = torch.LongTensor([word2id[token]]).to(device)
        right = "".join(right)

        if "+" in right or "<pad>" in right:
          accuracy += 0
          continue

        x = list(src[0].to('cpu').detach().numpy() )
        try:
            padded_idx_x = x.index(word2id["<pad>"])
        except ValueError:
            padded_idx_x = len(x)
        left = "".join(map(lambda c: str(id2word[c]), x[:padded_idx_x]))
        flag = ["F", "T"][eval(left) == int(right)]
        print("{:>7s} = {:>4s} :{}".format(left, right, flag))
        if flag == "T":
            accuracy += 1

print("Accuracy: {:.2f}".format(accuracy / len(test_loader)))

## 課題

1. `n_layers`（層数）や`heads`（Multi-Head Attentionのヘッド数）を変更し，精度や学習速度がどのように変化するか確認してみましょう．
2. 足し算だけでなく，色々な四則演算を実装しましょう．